# 11 - Main-text forest plots and absolute measures

Computes the absolute measures (rate difference and person-years of exposure per additional event, delta method) from the pooled hazard ratios and the drug-free reference strata, then draws the main forest plots (serious and all bleeding).

**Manuscript items:** Figs 2-5

**Inputs:** `RESULTS_DIR/*_summary.xlsx` and `RESULTS_DIR/reference_strata_*.xlsx` from notebooks 04-07

**Outputs:** updated `RESULTS_DIR/*_summary.xlsx`; `FIGURES_DIR/forest_plot_drug_{name,class,mech,num}.png`

*Derived from the original analysis notebook `all_hr_combine.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

In [ ]:
import time
import matplotlib.pyplot as plt
import os
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
from multiprocessing import Pool
from lifelines import CoxTimeVaryingFitter
import matplotlib.pyplot as plt
from matplotlib import gridspec

In [ ]:
def build_data(df):
    col_name = df.columns[0]
    data = []
    for _, row in df.iterrows():
        events, py = row["events/person_years"].split("/")
        data.append(
            dict(
                drug=row[col_name],
                severity=row["bleed_severity"],
                events=float(events),
                py=float(py),
                rate=float(events) / float(py) * 100,
                a_lo=float(row["ci_lower"]),
                a_hr=float(row["hr"]),
                a_hi=float(row["ci_upper"]),
                a_p=row["p"],
                c_lo=float(row["crude_ci_lower"]),
                c_hr=float(row["crude_hr"]),
                c_hi=float(row["crude_ci_upper"]),
                c_p=row["crude_p"],
            )
        )
    return data


# ---------------------------------------------------------------------------
# 2. Build the plot
# ---------------------------------------------------------------------------

def make_plot(data, col_name=None, lengend_loc=(1.2, 1.9)):
    # Group records by drug -> {severity: record}
    by_drug = {}
    for d in data:
        by_drug.setdefault(d["drug"], {})[d["severity"]] = d

    # Sort drugs by major-bleeding adjusted HR, descending
    drugs_sorted = sorted(
        by_drug.keys(),
        key=lambda dr: -by_drug[dr].get("major", {"a_hr": 0})["a_hr"],
    )

    # --- Layout sizing ---
    sub_row_h = 0.34     # vertical space per major/all sub-row
    drug_gap = 0.18      # extra gap between drug groups
    header_h = 1.3        # room for the spanning "Hazard ratio" heading + column headers

    n_drugs = len(drugs_sorted)
    total_h = n_drugs * (2 * sub_row_h + drug_gap) + header_h + 0.3
    fig_w = 13

    fig = plt.figure(figsize=(fig_w, total_h), facecolor="white")
    gs = gridspec.GridSpec(1, 2, width_ratios=[1.9, 1], wspace=0.02)

    ax_table = fig.add_subplot(gs[0])   # left: text table
    ax_forest = fig.add_subplot(gs[1])  # right: forest plot

    for ax in (ax_table, ax_forest):
        ax.set_ylim(0, total_h)
        ax.invert_yaxis()
        ax.axis("off")

    # Re-enable just the bottom/x-axis for the forest panel
    ax_forest.axis("on")
    for spine in ["top", "right", "left"]:
        ax_forest.spines[spine].set_visible(False)
    ax_forest.set_yticks([])

    # Column x-positions (axes-fraction coords within ax_table)
    col_drug  = 0.0
    col_sev   = 0.24
    col_evpy  = 0.50
    col_rate  = 0.75
    col_chr   = 1
    col_hr    = 1.29

    header_fs = 10.5
    row_fs = 9.6

    # x-axis (log scale) limits, based on the full range of CIs
    all_lo = [d["a_lo"] for d in data]
    all_hi = [d["a_hi"] for d in data]
    xmin = min(0.5, min(all_lo) * 0.8)
    xmax = max(2.0, max(all_hi) * 1.15)

    ax_forest.set_xscale("log")
    ax_forest.set_xlim(xmin, xmax)
    candidate_ticks = [0.5, 1, 2, 4, 8]
    ticks = [t for t in candidate_ticks if xmin <= t <= xmax] or [1]
    ax_forest.set_xticks(ticks)
    ax_forest.set_xticklabels([str(t) for t in ticks], fontsize=9)
    ax_forest.axvline(1.0, color="black", lw=0.8)
    ax_forest.tick_params(axis="x", length=3)

    # --- Spanning heading over the Crude HR and Adjusted HR columns ---
    span_y = header_h * 0.28
    span_mid = (col_chr + col_hr) / 2
    ax_table.text(
        span_mid, span_y, "Hazard ratio (95% CI)",
        fontsize=header_fs, fontweight="bold", va="center", ha="center",
    )
    # underline spanning just the two HR columns, roughly matching their text width
    span_half_width = (col_hr - col_chr) / 2 + 0.12
    ax_table.plot(
        [span_mid - span_half_width, span_mid + span_half_width],
        [span_y + header_h * 0.16, span_y + header_h * 0.16],
        color="black", lw=0.7, clip_on=False,
    )

    # --- Header row ---
    cur_y = header_h * 0.62
    if col_name =="Drug Mechanism":
        ax_table.text(col_drug, cur_y, "Drug\nMechanism", fontsize=header_fs, fontweight="bold", va="center", ha="center")
    else:
        ax_table.text(col_drug, cur_y, f"{col_name}", fontsize=header_fs, fontweight="bold", va="center", ha="center")
    ax_table.text(col_sev, cur_y, "Bleeding\nType", fontsize=header_fs, fontweight="bold", va="center", ha="center")
    ax_table.text(col_evpy, cur_y, "Events/\nperson-years", fontsize=header_fs, fontweight="bold", va="center", ha="center")
    ax_table.text(col_rate, cur_y, "Rate/100\nperson-yrs", fontsize=header_fs, fontweight="bold", va="center", ha="center")
    ax_table.text(col_chr, cur_y, "Crude HR\n(95% CI)", fontsize=header_fs, fontweight="bold", va="center", ha="center")
    ax_table.text(col_hr, cur_y, "Adjusted HR\n(95% CI)", fontsize=header_fs, fontweight="bold", va="center", ha="center")
    cur_y += header_h * 0.38
    ax_table.axhline(cur_y, color="black", lw=0.9, xmax=1.0)
    cur_y += sub_row_h * 0.5

    def sig_color(rec, base_color):
        """Return base_color if CI excludes 1 (significant), else grey."""
        if rec["a_lo"] > 1 or rec["a_hi"] < 1:
            return base_color
        return "#999999"

    major_color = "#c0392b"  # red
    all_color = "#1f4e9c"  # blue

    first_row_y = None
    last_row_y = None

    # --- Draw each drug group (major + minor rows together) ---
    for drug in drugs_sorted:
        grp = by_drug[drug]
        n_sub = len(grp)
        mid_y = cur_y + (n_sub - 1) * sub_row_h / 2
        ax_table.text(col_drug, mid_y, drug, fontsize=row_fs, va="center", ha="center", fontweight="medium")

        for sev, base_color, label in [
            ("major", major_color, "Serious"), ## label changed from "Major" to "Serious" as per feedback from co-authors
            ("all", all_color, "All"),
        ]:
            if sev not in grp:
                continue
            d = grp[sev]
            yy = cur_y
            if first_row_y is None:
                first_row_y = yy
            last_row_y = yy

            # Text columns
            ax_table.text(col_sev, yy, label, fontsize=row_fs, va="center", ha="center", color=base_color)
            ax_table.text(col_evpy, yy, f"{int(d['events'])}/{int(d['py'])}", fontsize=row_fs, va="center", ha="center")
            ax_table.text(col_rate, yy, f"{d['rate']:.1f}", fontsize=row_fs, va="center", ha="center")
            ax_table.text(
                col_chr, yy,
                f"{d['c_hr']:.2f} ({d['c_lo']:.2f}-{d['c_hi']:.2f})",
                fontsize=row_fs, va="center", ha="center",
            )
            ax_table.text(
                col_hr, yy,
                f"{d['a_hr']:.2f} ({d['a_lo']:.2f}-{d['a_hi']:.2f})",
                fontsize=row_fs, va="center", ha="center",
            )

            # Forest plot: CI line + point estimate marker
            color = sig_color(d, base_color)
            marker = "s" if sev == "major" else "o"

            ax_forest.plot([d["a_lo"], d["a_hi"]], [yy, yy], color=color, lw=1.3, solid_capstyle="butt")
            ax_forest.plot(d["a_hr"], yy, marker=marker, color=color, markersize=6, zorder=3)
            # CI whisker caps
            ax_forest.plot([d["a_lo"], d["a_lo"]], [yy - 0.08, yy + 0.08], color=color, lw=1.0)
            ax_forest.plot([d["a_hi"], d["a_hi"]], [yy - 0.08, yy + 0.08], color=color, lw=1.0)

            cur_y += sub_row_h

        cur_y += drug_gap

    ax_table.set_xlim(0, 1.45)

    # --- Trim blank top/bottom margin from the forest panel only ---
    # (the table panel keeps its full height for the header; the forest
    # panel's box is shrunk to exactly bound the first->last row)
    row_top = first_row_y - sub_row_h / 2
    row_bottom = last_row_y + sub_row_h / 2

    pos = ax_forest.get_position()
    frac_top = 1 - row_top / total_h       # inverted axis: 0 is at the top
    frac_bottom = 1 - row_bottom / total_h
    new_top = pos.y0 + pos.height * frac_top
    new_bottom = pos.y0 + pos.height * frac_bottom
    ax_forest.set_position([pos.x0, new_bottom, pos.width, new_top - new_bottom])
    ax_forest.set_ylim(row_bottom, row_top)  # keeps the inverted orientation

    # Legend distinguishing major vs all bleeding
    ax_forest.plot([], [], marker="s", color=major_color, lw=1.3, label="Serious bleeding")
    ax_forest.plot([], [], marker="o", color=all_color, lw=1.3, label="All bleedings")
    ax_forest.legend(loc="upper right", fontsize=9, frameon=False, bbox_to_anchor=lengend_loc)

    ax_forest.set_xlabel("Adjusted hazard ratio (95% CI), log scale", fontsize=10)

    #return the figure object so the caller can save it or display it
    return fig

In [ ]:
hr_major_drugs = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_name_major_summary.xlsx"), dtype=str)
hr_minor_drugs = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_name_minor_summary.xlsx"), dtype=str)
hr_all_drugs = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_name_all_summary.xlsx"), dtype=str)

hr_major_drugs = hr_major_drugs[hr_major_drugs["drug_class"] !="Enoxaparin"]
hr_minor_drugs = hr_minor_drugs[hr_minor_drugs["drug_class"] !="Enoxaparin"]
hr_all_drugs = hr_all_drugs[hr_all_drugs["drug_class"] !="Enoxaparin"]

In [ ]:
columns_mapping = {
    "Patient Counts": "patient_counts",
    "HR": "hr",
    "CI Lower": "ci_lower",
    "CI Upper": "ci_upper",
    "p": "p",
    "No of Events": "no_of_events",
    "Crude HR":"crude_hr",
    "Crude CI Lower":"crude_ci_lower",
    "Crude CI Upper":"crude_ci_upper",
    "Crude p":"crude_p",
}


hr_major_drugs.rename(columns=columns_mapping, inplace=True)
hr_minor_drugs.rename(columns=columns_mapping, inplace=True)
hr_all_drugs.rename(columns=columns_mapping, inplace=True)


In [ ]:
columns_to_consider = ["drug_class", "hr", "ci_lower", "ci_upper", "p", "events/person_years","crude_hr","crude_ci_lower", "crude_ci_upper", "crude_p"]

hr_major_drugs = hr_major_drugs[columns_to_consider]
hr_all_drugs = hr_all_drugs[columns_to_consider]

hr_major_drugs_merged = hr_major_drugs.merge(hr_all_drugs, on="drug_class", suffixes=("_major", "_all"))
hr_major_drugs_merged = pd.concat(
    [
        hr_major_drugs.assign(bleed_severity="major"),
        hr_all_drugs.assign(bleed_severity="all"),
    ],
    ignore_index=True,
 )

hr_major_drugs_merged["bleed_severity"] = pd.Categorical(
    hr_major_drugs_merged["bleed_severity"],
    categories=["major", "all"],
    ordered=True,
 )
hr_major_drugs_merged = hr_major_drugs_merged.sort_values(["drug_class", "bleed_severity"],kind="stable").reset_index(drop=True)

In [ ]:
# Assumes hr_major_drugs_merged already exists in your notebook/session
data = build_data(hr_major_drugs_merged)
drug_name_fig = make_plot(data, col_name="Drug Name", lengend_loc=(1.2, 1.09))

OUTPUT_PNG = os.path.join(FIGURES_DIR, "forest_plot_drug_name.png")
drug_name_fig.savefig(OUTPUT_PNG, dpi=200, bbox_inches="tight", facecolor="white", transparent=False)

In [ ]:
columns_mapping = {
    "HR":"hr",
    "CI Lower":"ci_lower",
    "CI Upper":"ci_upper",
    "Crude HR":"crude_hr",
    "Crude CI Lower":"crude_ci_lower",
    "Crude CI Upper":"crude_ci_upper",
    "Crude p":"crude_p"
    
}

In [ ]:
hr_major_drug_class = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_class_major_summary.xlsx"), dtype=str)
hr_all_drug_class = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_class_all_summary.xlsx"), dtype=str)
hr_major_drug_class.rename(columns=columns_mapping, inplace=True)
hr_all_drug_class.rename(columns=columns_mapping, inplace=True)

columns_to_consider = ["drug_class", "hr", "ci_lower", "ci_upper", "p", "events/person_years","crude_hr","crude_ci_lower", "crude_ci_upper", "crude_p"]

hr_major_drug_class = hr_major_drug_class[columns_to_consider]
hr_all_drug_class = hr_all_drug_class[columns_to_consider]

hr_drug_class_merged = hr_major_drug_class.merge(hr_all_drug_class, on="drug_class", suffixes=("_major", "_all"))
hr_drug_class_merged = pd.concat(
    [
        hr_major_drug_class.assign(bleed_severity="major"),
        hr_all_drug_class.assign(bleed_severity="all"),
    ],
    ignore_index=True,
 )

hr_drug_class_merged["bleed_severity"] = pd.Categorical(
    hr_drug_class_merged["bleed_severity"],
    categories=["major", "all"],
    ordered=True,
 )
hr_drug_class_merged = hr_drug_class_merged.sort_values(["drug_class", "bleed_severity"],kind="stable").reset_index(drop=True)

In [ ]:
data_drug_class = build_data(hr_drug_class_merged)
OUTPUT_PNG = os.path.join(FIGURES_DIR, "forest_plot_drug_class.png")
drug_class_fig = make_plot(data_drug_class, col_name="Drug Class", lengend_loc=(1.2, 1.2))
drug_class_fig.savefig(OUTPUT_PNG, dpi=200, bbox_inches="tight", facecolor="white", transparent=False)

In [ ]:
hr_major_drug_num = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_num_major_summary.xlsx"), dtype=str)
hr_all_drug_num = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_num_all_summary.xlsx"), dtype=str)

hr_major_drug_num.rename(columns=columns_mapping, inplace=True)
hr_all_drug_num.rename(columns=columns_mapping, inplace=True)

columns_to_consider = ["drug_class", "hr", "ci_lower", "ci_upper", "p", "events/person_years","crude_hr","crude_ci_lower", "crude_ci_upper", "crude_p"]

hr_major_drug_num = hr_major_drug_num[columns_to_consider]
hr_all_drug_num = hr_all_drug_num[columns_to_consider]

hr_drug_num_merged = hr_major_drug_num.merge(hr_all_drug_num, on="drug_class", suffixes=("_major", "_all"))
hr_drug_num_merged = pd.concat(
    [
        hr_major_drug_num.assign(bleed_severity="major"),
        hr_all_drug_num.assign(bleed_severity="all"),
    ],
    ignore_index=True,
 )

hr_drug_num_merged["bleed_severity"] = pd.Categorical(
    hr_drug_num_merged["bleed_severity"],
    categories=["major", "all"],
    ordered=True,
 )
hr_drug_num_merged = hr_drug_num_merged.sort_values(["drug_class", "bleed_severity"],kind="stable").reset_index(drop=True)
hr_drug_num_merged.rename(columns={"drug_class": "No. of Drugs"}, inplace=True)
hr_drug_num_merged["No. of Drugs"].replace({
    "1_Drug": "1 Drug",
    "2_Drugs": "2 Drugs",
    "3_or_more_Drugs": "≥ 3 Drugs",
}, inplace=True)

In [ ]:
data_drug_num = build_data(hr_drug_num_merged)
OUTPUT_PNG = os.path.join(FIGURES_DIR, "forest_plot_drug_num.png")

drug_num_fig = make_plot(data_drug_num, col_name="No. of drugs", lengend_loc=(1.2, 1.6))
drug_num_fig.savefig(OUTPUT_PNG, dpi=200, bbox_inches="tight", facecolor="white", transparent=False)

In [ ]:
hr_major_drug_mech = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_pk_pd_major_summary.xlsx"), dtype=str)
hr_all_drug_mech = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_pk_pd_all_summary.xlsx"), dtype=str)

hr_major_drug_mech.rename(columns=columns_mapping, inplace=True)
hr_all_drug_mech.rename(columns=columns_mapping, inplace=True)

columns_to_consider = ["drug_class", "hr", "ci_lower", "ci_upper", "p", "events/person_years","crude_hr","crude_ci_lower", "crude_ci_upper", "crude_p"]

hr_major_drug_mech = hr_major_drug_mech[columns_to_consider]
hr_all_drug_mech = hr_all_drug_mech[columns_to_consider]

hr_drug_mech_merged = hr_major_drug_mech.merge(hr_all_drug_mech, on="drug_class", suffixes=("_major", "_all"))
hr_drug_mech_merged = pd.concat(
    [
        hr_major_drug_mech.assign(bleed_severity="major"),
        hr_all_drug_mech.assign(bleed_severity="all"),
    ],
    ignore_index=True,
 )

hr_drug_mech_merged["bleed_severity"] = pd.Categorical(
    hr_drug_mech_merged["bleed_severity"],
    categories=["major", "all"],
    ordered=True,
 )
#hr_drug_mech_merged = hr_drug_mech_merged.sort_values(["drug_class", "bleed_severity"],kind="stable").reset_index(drop=True)

In [ ]:
#hr_drug_mech_merged.rename(columns={"drug_class": "PK/PD mechanism"}, inplace=True)
hr_drug_mech_merged["drug_class"].replace({
    "p_k_drug": "PK Mechanism",
    "p_d_drug": "PD Mechanism",
}, inplace=True)
data_drug_mech = build_data(hr_drug_mech_merged)
OUTPUT_PNG = os.path.join(FIGURES_DIR, "forest_plot_drug_mech.png")
drug_mech_fig = make_plot(data_drug_mech, col_name="Drug Mechanism", lengend_loc=(1.2, 1.8))
drug_mech_fig.savefig(OUTPUT_PNG, dpi=200, bbox_inches="tight", facecolor="white", transparent=False)

In [ ]:
DEFAULT_COLUMNS_MAPPING = {
    "Patient Counts": "patient_counts",
    "HR": "hr",
    "CI Lower": "ci_lower",
    "CI Upper": "ci_upper",
    "p": "p",
    "No of Events": "no_of_events",
    "Crude HR": "crude_hr",
    "Crude CI Lower": "crude_ci_lower",
    "Crude CI Upper": "crude_ci_upper",
    "Crude p": "crude_p",
}

KEEP_COLUMNS = [
    "drug_class", "patient_counts", "events/person_years", "events_per_100py",
    "hr", "ci_lower", "ci_upper", "p", "NNH", "NNH_CI_lower", "NNH_CI_upper",
]

# The drug-free reference stratum behind every NNH in the table. Carried through
# so the supplementary footnotes can quote it instead of leaving it unreported.
# Dropped silently for summary files written before the corrected
# calculate_nnh(), so this cell still runs against older results.
REF_COLUMNS = ["ref_events", "ref_person_years", "rate_unexposed_per_100py"]


def build_drug_num_table(
    major_path=os.path.join(RESULTS_DIR, "hr_drug_num_major_summary.xlsx"),
    minor_path=os.path.join(RESULTS_DIR, "hr_drug_num_minor_summary.xlsx"),
    all_path=os.path.join(RESULTS_DIR, "hr_drug_num_all_summary.xlsx"),
    columns_mapping=None,
    keep_columns=None,
    bleeding_order=("Severe", "Minor", "All"),
):
    """Combine severe/minor/all HR summaries into one long table,
    ordered by descending severe-bleeding HR."""
    columns_mapping = columns_mapping or DEFAULT_COLUMNS_MAPPING
    keep_columns = keep_columns or KEEP_COLUMNS

    # Load + normalise each table (paths pair with bleeding_order)
    frames = {}
    for label, path in zip(bleeding_order, [major_path, minor_path, all_path]):
        d = pd.read_excel(path, dtype=str)
        d.rename(columns=columns_mapping, inplace=True)
        cols = list(keep_columns) + [c for c in REF_COLUMNS if c in d.columns]
        frames[label] = d[cols]

    # Interleave so each drug's Severe/Minor/All rows sit together
    combined = pd.concat(frames, names=["bleeding type", "drug"]).swaplevel()
    drug_positions = frames[bleeding_order[0]].index.tolist()
    combined = combined.reindex(
        pd.MultiIndex.from_product(
            [drug_positions, list(bleeding_order)],
            names=["drug", "bleeding"],
        )
    )
    combined = combined.reset_index().drop(columns=["drug"])

    df = combined.copy()
    df["bleeding"] = pd.Categorical(df["bleeding"], list(bleeding_order), ordered=True)

    # Drug order = severe rows sorted by HR desc, then any drugs not in severe
    drug_order = (
        df[df["bleeding"] == bleeding_order[0]]
        .assign(_hr=lambda t: pd.to_numeric(t["hr"], errors="coerce"))
        .sort_values("_hr", ascending=False)["drug_class"]
        .tolist()
    )
    drug_order += [d for d in df["drug_class"].unique() if d not in drug_order]

    df["drug_class"] = pd.Categorical(df["drug_class"], drug_order, ordered=True)
    return df.sort_values(["drug_class", "bleeding"])

In [ ]:
# ===========================================================================
# Recompute the absolute measures WITHOUT refitting
# ===========================================================================
# The absolute measures (rate difference and person-years of exposure per
# additional event, 'NNH') are computed without refitting, from
#   (a) HR, its CI and the Barnard-Rubin df already in those files, and
#   (b) the drug-free reference stratum, which needs only the interval frame
#       (major_df / minor_df / all_df) - no Cox fit, no MICE.
# Run the fitting notebook up to the cell that builds *_df, then run this.
#
# The reference rate r0 is the rate in person-time with NO interacting drug
# active; the `rate_unexposed_per_100py` column is (re)written below from it.

from scipy import stats


def reference_from_frame(df, all_terms=None):
    """Events, person-years and rate per 100 PY in drug-free person-time."""
    if "concomitant_drugs_ls" in df.columns:
        mask = df["concomitant_drugs_ls"].map(
            lambda v: len(v) == 0 if isinstance(v, (list, tuple, set)) else True).to_numpy(bool)
    elif all_terms:
        cols = [t for t in all_terms if t in df.columns]
        mask = (df[cols].sum(axis=1) == 0).to_numpy(bool)
    else:
        raise ValueError("pass all_terms, or keep concomitant_drugs_ls on the frame")
    sub = df.loc[mask]
    days = ((sub["end_date"] - sub["start_date"]).dt.total_seconds() / 86400.0).clip(lower=0)
    py = float(days.sum()) / 365.25
    ev = int(sub.loc[sub["event_flag"] == 1, "patid"].nunique())
    return ev, py, (ev / py * 100 if py > 0 else np.nan)


def recompute_absolute_measures(path, ref_events, ref_py, out_path=None):
    """Rewrite the NNH columns of one summary file using RD = r0 x (HR - 1)."""
    d = pd.read_excel(path)
    ren = {"HR": "hr", "CI Lower": "ci_lower", "CI Upper": "ci_upper"}
    d = d.rename(columns={k: v for k, v in ren.items() if k in d.columns})

    hr = pd.to_numeric(d["hr"], errors="coerce")
    lo = pd.to_numeric(d["ci_lower"], errors="coerce")
    hi = pd.to_numeric(d["ci_upper"], errors="coerce")

    # Recover SE(log HR) from the printed interval, using the same quantile the
    # interval was built with (t on the Barnard-Rubin df after Rubin pooling).
    dof = pd.to_numeric(d["MI df"], errors="coerce") if "MI df" in d.columns else None
    crit = (stats.t.ppf(0.975, dof) if dof is not None else np.full(len(d), 1.959963985))
    crit = pd.Series(np.where(np.isfinite(crit), crit, 1.959963985), index=d.index)
    se_log_hr = (np.log(hi) - np.log(lo)) / (2 * crit)

    r0 = ref_events / ref_py                       # events per person-year
    var_log_r0 = 1.0 / ref_events
    rd = r0 * (hr - 1.0)
    se_rd = r0 * np.sqrt((hr - 1.0) ** 2 * var_log_r0 + hr ** 2 * se_log_hr ** 2)
    rd_lo, rd_hi = rd - crit * se_rd, rd + crit * se_rd

    inv = lambda x: np.where(x > 0, 1.0 / x, np.nan)
    d["ref_events"] = ref_events
    d["ref_person_years"] = round(ref_py, 2)
    d["rate_unexposed_per_100py"] = round(r0 * 100, 3)   # overwrites the old value
    d["rate_diff_per_1000py"] = (rd * 1000).round(3)
    d["rate_diff_per_1000py_lower"] = (rd_lo * 1000).round(3)
    d["rate_diff_per_1000py_upper"] = (rd_hi * 1000).round(3)
    d["NNH"] = pd.Series(inv(rd), index=d.index).round(1).where(rd > 0, "Not estimable")
    d["NNH_CI_lower"] = pd.Series(inv(rd_hi), index=d.index).round(1)
    d["NNH_CI_upper"] = pd.Series(inv(rd_lo), index=d.index).round(1)
    if "CI_crosses_null" in d.columns:      # else it contradicts the new bounds
        d["CI_crosses_null"] = (rd_lo <= 0) & (rd_hi >= 0)

    d = d.rename(columns={v: k for k, v in ren.items() if v in d.columns})
    d.to_excel(out_path or path, index=False)
    return d


In [ ]:
# ---- Driver: rebuild the absolute measures in place -----------------------
# The drug-free reference stratum cannot be recovered from results/*.xlsx - it
# exists only in the interval frames. Each fitting notebook therefore computes
# its own and writes it out; this cell reads them back.
#
# There is ONE FILE PER ANALYSIS, not one overall. "No interacting drug active"
# means something different for 18 named drugs, for 8 drug classes, for the
# PK/PD mechanism groups and for the drug-count categories, so the four analyses
# have four different reference strata and must not share one.

GROUPS = {                       # group: (summary-file template, strata file)
    "drug_name":  ("hr_drug_name_{}_summary.xlsx",  "reference_strata_drug_name.xlsx"),
    "drug_class": ("hr_drug_class_{}_summary.xlsx", "reference_strata_drug_class.xlsx"),
    "drug_num":   ("hr_drug_num_{}_summary.xlsx",   "reference_strata_drug_num.xlsx"),
    "pk_pd":      ("hr_drug_pk_pd_{}_summary.xlsx", "reference_strata_pk_pd.xlsx"),
}
_SUFFIX = {"Serious": "major", "Minor": "minor", "All": "all"}

for group, (tmpl, ref_file) in GROUPS.items():
    try:
        _ref = pd.read_excel(RESULTS_DIR + "/" + ref_file)
    except FileNotFoundError:
        print(f"[{group}] skipped - results/{ref_file} not found. Run the step-1 "
              f"cell in the matching fitting notebook first.")
        continue

    reference = {r.outcome: (int(r.ref_events), float(r.ref_person_years))
                 for r in _ref.itertuples()}
    missing = set(_SUFFIX) - set(reference)
    if missing:
        raise ValueError(f"{ref_file} is missing {sorted(missing)} - all three "
                         "outcomes are needed, they have different strata.")

    print(f"[{group}] using {ref_file}")
    for outcome, (ev, py) in reference.items():
        path = RESULTS_DIR + "/" + tmpl.format(_SUFFIX[outcome])
        try:
            recompute_absolute_measures(path, ev, py)
            print(f"    rewrote {path}  (r0 = {ev/py*100:.3f} per 100 PY)")
        except FileNotFoundError:
            print(f"    skipped {path} (not found)")


In [ ]:
# ===========================================================================
# Reference stratum (drug-free person-time) for the supplementary footnotes
# ===========================================================================
# The absolute measure is RD = r0 x (HR - 1), where r0 is the event
# rate in person-time with NO interacting drug active. Each fitting
# notebook writes that stratum into every summary file as ref_events,
# ref_person_years and rate_unexposed_per_100py.
#
# r0 is NOT the same for the three outcomes: follow-up ends at the first event
# OF THAT TYPE, so the drug-free person-time in the serious, minor and
# all-bleeding cohorts differs. Tables S9, S10, S12 and S13 therefore need three
# reference strata each, not one.

REF_COLS = ["ref_events", "ref_person_years", "rate_unexposed_per_100py"]


def reference_strata(major_path, minor_path, all_path,
                     labels=("Serious", "Minor", "All"),
                     cohort_rates_per_100py=None):
    """One reference stratum per outcome, read back from the summary files.

    Every row of a given summary file is a contrast against the same drug-free
    person-time, so the three REF_COLS must be constant down each file. If they
    are not, the reference mask was applied per-drug rather than once per table
    and the fitting notebook needs re-checking before these numbers are used.
    """
    rows = []
    for label, path in zip(labels, (major_path, minor_path, all_path)):
        d = pd.read_excel(path)
        missing = [c for c in REF_COLS if c not in d.columns]
        if missing:
            raise KeyError(
                f"{path} has no {missing}. This file predates the corrected "
                "calculate_nnh() - rerun the fitting notebook before building "
                "the footnotes.")

        uniq = d[REF_COLS].apply(pd.to_numeric, errors="coerce").drop_duplicates()
        if len(uniq) != 1:
            raise ValueError(
                f"{path}: reference stratum is not constant across rows "
                f"({len(uniq)} distinct values). Every hazard ratio in a table "
                "must share one reference - check that build_bleeding_summary "
                "received all_terms=drug_terms.")

        ev, py, rate = uniq.iloc[0][REF_COLS]
        row = {"outcome": label, "ref_events": int(ev),
               "ref_person_years": float(py), "rate_per_100py": float(rate)}

        # Sanity check: every exposed stratum has a rate above the cohort mean,
        # so the drug-free rate must sit BELOW it.
        if cohort_rates_per_100py and label in cohort_rates_per_100py:
            cohort = float(cohort_rates_per_100py[label])
            row["cohort_rate_per_100py"] = cohort
            row["passes_check"] = row["rate_per_100py"] < cohort
        rows.append(row)

    out = pd.DataFrame(rows)
    if "passes_check" in out.columns and not out["passes_check"].all():
        bad = out.loc[~out["passes_check"], "outcome"].tolist()
        print(f"  FAILED sanity check for {bad}: the drug-free rate is at or "
              "above the cohort-wide rate, which is impossible. Do not use "
              "these numbers - the reference mask is not being applied.")
    return out


def footnote_reference_sentence(ref_table, decimals=2):
    """The reference-stratum sentence used in the supplementary table
    footnotes, covering all three outcomes in one sentence."""
    parts = [
        f"{r.outcome.lower()} bleeding, {r.ref_events:,} events over "
        f"{r.ref_person_years:,.0f} person-years, {r.rate_per_100py:.{decimals}f} "
        "per 100 person-years"
        for r in ref_table.itertuples()]
    return ("r0 is the event rate observed in person-time during which no "
            "interacting drug was active, that is the reference level of the "
            "mutually adjusted Cox models. Because follow-up ends at the first "
            "event of each type, the reference stratum differs between outcomes: "
            + "; ".join(parts) + ".")


In [ ]:
# ---- Reference strata for the supplementary footnotes ---------------------
# One call per analysis: Table S9 uses drug_name, S10 drug_class, S12 pk_pd,
# S13 drug_num. Each has its own reference, so each gets its own footnote.
for _group, _file in [("drug_name","reference_strata_drug_name.xlsx"),
                      ("drug_class","reference_strata_drug_class.xlsx"),
                      ("pk_pd","reference_strata_pk_pd.xlsx"),
                      ("drug_num","reference_strata_drug_num.xlsx")]:
    try:
        _t = pd.read_excel(RESULTS_DIR + "/" + _file)
    except FileNotFoundError:
        print(f"[{_group}] {_file} not found - run the matching step-1 cell."); continue
    print(f"\n=== {_group} ===")
    display(_t)
    print(footnote_reference_sentence(_t.rename(columns={"rate_per_100py":"rate_per_100py"})))


In [ ]:
def _fmt(v, dp=1):
    """Numbers to `dp` places; NaN -> 'NA'; text like 'Not estimable' passes through."""
    if pd.isna(v):
        return "NA"
    try:
        return f"{float(v):.{dp}f}"
    except (TypeError, ValueError):
        return str(v).strip()


def merge_ci(df, point, lo, hi, name, dp=1):
    """Collapse a point estimate and its two CI columns into one 'x (l to h)' column,
    in the position the point estimate held."""
    def row(r):
        p, l, h = _fmt(r[point], dp), _fmt(r[lo], dp), _fmt(r[hi], dp)
        return p if l == "NA" and h == "NA" else f"{p} ({l} to {h})"
    out = df.copy()
    out.insert(out.columns.get_loc(point), name, out.apply(row, axis=1))
    return out.drop(columns=[point, lo, hi])


def fmt_p(v, dp=2):
    """P values to `dp` places, with '<0.001' below that.

    Guards the case plain rounding gets wrong: p = 0.0026 to two places prints
    '0.00', which reads as zero. Anything that would round to 0.00 but is not
    below 0.001 is shown as '<0.01' instead.
    """
    v = pd.to_numeric(v, errors="coerce")
    if pd.isna(v):
        return "NA"
    if v < 0.001:
        return "<0.001"
    if round(v, dp) == 0:
        return f"<{10**-dp:.{dp}f}"
    return f"{v:.{dp}f}"


drug_name = build_drug_num_table(
    major_path=os.path.join(RESULTS_DIR, "hr_drug_name_major_summary.xlsx"),
    minor_path=os.path.join(RESULTS_DIR, "hr_drug_name_minor_summary.xlsx"),
    all_path=os.path.join(RESULTS_DIR, "hr_drug_name_all_summary.xlsx"),)
drug_name = drug_name.drop(
    columns=["ref_events", "ref_person_years", "rate_unexposed_per_100py"], errors="ignore")
drug_name = merge_ci(drug_name, "NNH", "NNH_CI_lower", "NNH_CI_upper",
                     "Person-years of exposure per additional event  (95% CI)", dp=1)
drug_name = merge_ci(drug_name, "hr", "ci_lower", "ci_upper",
                     "Adjusted hazard ratio (95% CI)", dp=2)
drug_name["p"] = drug_name["p"].apply(fmt_p)
drug_name

In [ ]:
drug_class = build_drug_num_table(
    major_path=os.path.join(RESULTS_DIR, "hr_drug_class_major_summary.xlsx"),
    minor_path=os.path.join(RESULTS_DIR, "hr_drug_class_minor_summary.xlsx"),
    all_path=os.path.join(RESULTS_DIR, "hr_drug_class_all_summary.xlsx"),)
drug_class = drug_class.drop(columns=["ref_events", "ref_person_years", "rate_unexposed_per_100py"], errors="ignore")
drug_class = merge_ci(drug_class, "NNH", "NNH_CI_lower", "NNH_CI_upper",
                      "Person-years of exposure per additional event  (95% CI)", dp=1)
drug_class = merge_ci(drug_class, "hr", "ci_lower", "ci_upper",
                      "Adjusted hazard ratio (95% CI)", dp=2)
drug_class["p"] = drug_class["p"].apply(fmt_p)
drug_class

In [ ]:
drug_num = build_drug_num_table(
    major_path=os.path.join(RESULTS_DIR, "hr_drug_num_major_summary.xlsx"),
    minor_path=os.path.join(RESULTS_DIR, "hr_drug_num_minor_summary.xlsx"),
    all_path=os.path.join(RESULTS_DIR, "hr_drug_num_all_summary.xlsx"),
)

drug_num = drug_num.drop(columns=["ref_events", "ref_person_years", "rate_unexposed_per_100py"], errors="ignore")
drug_num = merge_ci(drug_num, "NNH", "NNH_CI_lower", "NNH_CI_upper",
                    "Person-years of exposure per additional event  (95% CI)", dp=1)
drug_num = merge_ci(drug_num, "hr", "ci_lower", "ci_upper",
                    "Adjusted hazard ratio (95% CI)", dp=2)
drug_num["p"] = drug_num["p"].apply(fmt_p)  
drug_num

In [ ]:
drug_mech_hr_data = build_drug_num_table(
    major_path=os.path.join(RESULTS_DIR, "hr_drug_pk_pd_major_summary.xlsx"),
    minor_path=os.path.join(RESULTS_DIR, "hr_drug_pk_pd_minor_summary.xlsx"),
    all_path=os.path.join(RESULTS_DIR, "hr_drug_pk_pd_all_summary.xlsx"),
)
drug_mech_hr_data = drug_mech_hr_data.drop(columns=["ref_events", "ref_person_years", "rate_unexposed_per_100py"], errors="ignore")
drug_mech_hr_data = merge_ci(drug_mech_hr_data, "NNH", "NNH_CI_lower", "NNH_CI_upper",
                             "Person-years of exposure per additional event  (95% CI)", dp=1)
drug_mech_hr_data = merge_ci(drug_mech_hr_data, "hr", "ci_lower", "ci_upper",
                             "Adjusted hazard ratio (95% CI)", dp=2)
drug_mech_hr_data["p"] = drug_mech_hr_data["p"].apply(fmt_p)
drug_mech_hr_data